# Maths Verification Notebook

A companion to the [Report card](../README.md) tracker for Advanced Higher Maths.

The idea is the same as running a problem through a CAS like *Derive*: instead of trusting an
answer you worked out by hand, you re-derive it symbolically here and let SymPy confirm the two
match. Use it after a past paper or homework set — paste in the question's expression and your
worked answer, run the cell, and get a clear **PASS/FAIL** instead of a guess.

**How it works**

- `verify(expression, claimed_answer)` tries to prove the two are symbolically identical
  (`simplify`, then `equals`), and falls back to checking numerically at several random points
  when a symbolic proof doesn't resolve — this catches most transcription/sign/simplification
  mistakes without needing a fully closed-form proof for messy trig or log expressions.
- Each syllabus section below (differentiation, integration, equations, sequences & series,
  complex numbers, vectors) has worked examples showing the pattern, plus a blank template at
  the end for your own problems.


In [1]:
import sympy as sp
from sympy import (
    symbols, Symbol, diff, integrate, simplify, solve, solveset, Eq,
    sqrt, sin, cos, tan, exp, log, pi, I, re, im, Abs, arg, oo, S,
    Sum, summation, Rational, nsimplify, factor, expand, together, cancel
)
import random

x, y, z, t, n, k, theta = symbols('x y z t n k theta', real=True)

def verify(expr_a, expr_b, *, label="", domain_vars=None, samples=25, tol=1e-9):
    """Check whether expr_a and expr_b are the same expression.

    Tries an exact symbolic proof first (difference simplifies to 0, or .equals()).
    If SymPy can't decide symbolically (common with trig/log/piecewise forms), falls
    back to substituting random numeric values for every free symbol and checking the
    two sides agree everywhere they're both defined.
    """
    expr_a = sp.sympify(expr_a)
    expr_b = sp.sympify(expr_b)
    diff_expr = sp.together(expr_a - expr_b)

    verdict = None
    method = None

    simplified = sp.simplify(diff_expr)
    if simplified == 0:
        verdict, method = True, "symbolic simplify"
    else:
        eq = expr_a.equals(expr_b)
        if eq is True:
            verdict, method = True, "symbolic .equals()"
        elif eq is False:
            verdict, method = False, "symbolic .equals()"

    if verdict is None:
        free = sorted((expr_a.free_symbols | expr_b.free_symbols), key=str)
        free = domain_vars if domain_vars else free
        checks, mismatches = 0, 0
        for _ in range(samples):
            subs = {s: Rational(random.randint(-500, 500), random.randint(1, 37)) for s in free}
            try:
                va = complex(expr_a.evalf(subs=subs))
                vb = complex(expr_b.evalf(subs=subs))
            except (TypeError, ValueError):
                continue
            if any(map(lambda v: v != v, (va.real, va.imag, vb.real, vb.imag))):
                continue
            checks += 1
            if abs(va - vb) > tol * max(1, abs(va), abs(vb)):
                mismatches += 1
        if checks == 0:
            verdict, method = None, "could not evaluate numerically"
        else:
            verdict, method = (mismatches == 0), f"numeric check ({checks} random points)"

    tag = "PASS" if verdict is True else ("FAIL" if verdict is False else "INCONCLUSIVE")
    icon = {"PASS": "\u2713", "FAIL": "\u2717", "INCONCLUSIVE": "?"}[tag]
    header = f"{icon} {tag}" + (f" — {label}" if label else "")
    print(header)
    print(f"  your answer:   {expr_b}")
    print(f"  derived form:  {sp.radsimp(sp.simplify(expr_a))}")
    print(f"  method:        {method}")
    return verdict


## Differentiation

Differentiate the expression yourself, then check your answer against SymPy's derivative.

In [2]:
# Example: differentiate f(x) = x^3 * sin(2x)
f = x**3 * sin(2*x)
my_answer = 3*x**2*sin(2*x) + 2*x**3*cos(2*x)

derived = diff(f, x)
verify(derived, my_answer, label="d/dx[x^3 sin(2x)]")


✓ PASS — d/dx[x^3 sin(2x)]
  your answer:   2*x**3*cos(2*x) + 3*x**2*sin(2*x)
  derived form:  x**2*(2*x*cos(2*x) + 3*sin(2*x))
  method:        symbolic simplify


True

In [3]:
# Example: a wrong answer on purpose, to show what a FAIL looks like
f2 = (x**2 + 1) / (x - 1)
my_wrong_answer = (2*x*(x - 1) - (x**2 + 1)) / (x - 1)**2 + 1  # deliberate slip: +1 shouldn't be there

derived2 = diff(f2, x)
verify(derived2, my_wrong_answer, label="d/dx[(x^2+1)/(x-1)] (has a deliberate mistake)")


✗ FAIL — d/dx[(x^2+1)/(x-1)] (has a deliberate mistake)
  your answer:   1 + (-x**2 + 2*x*(x - 1) - 1)/(x - 1)**2


  derived form:  (-x**2 + 2*x*(x - 1) - 1)/(x - 1)**2
  method:        symbolic .equals()


False

## Integration

Same pattern — integrate by hand, check against SymPy's antiderivative. (Integration answers differ by a constant, so compare derivatives to sidestep the `+ C` question, or just check `equals()` up to a constant as shown below.)

In [4]:
# Example: integrate x * exp(x^2)
g = x * exp(x**2)
my_answer = sp.Rational(1, 2) * exp(x**2)  # + C, omitted

derived_antideriv = integrate(g, x)
# Compare derivatives of both sides instead of the antiderivatives themselves,
# so an arbitrary "+ C" doesn't cause a false FAIL.
verify(diff(derived_antideriv, x), diff(my_answer, x), label="∫ x e^(x^2) dx")


✓ PASS — ∫ x e^(x^2) dx
  your answer:   x*exp(x**2)
  derived form:  x*exp(x**2)
  method:        symbolic simplify


True

In [5]:
# Definite integral — no +C ambiguity, so just check the value directly
h = sin(x)**2
a_lo, a_hi = 0, pi
my_answer_def = pi / 2

derived_def = integrate(h, (x, a_lo, a_hi))
verify(derived_def, my_answer_def, label="∫₀^π sin²(x) dx")


✓ PASS — ∫₀^π sin²(x) dx
  your answer:   pi/2
  derived form:  pi/2
  method:        symbolic simplify


True

## Equations & algebra

Solve the equation yourself, then compare your root set to SymPy's.

In [6]:
# Example: solve x^3 - 6x^2 + 11x - 6 = 0
equation = x**3 - 6*x**2 + 11*x - 6
my_roots = {1, 2, 3}

derived_roots = set(solve(Eq(equation, 0), x))
match = derived_roots == my_roots
icon = "\u2713" if match else "\u2717"
print(f"{icon} {'PASS' if match else 'FAIL'} — roots of x^3 - 6x^2 + 11x - 6 = 0")
print(f"  your roots:    {sorted(my_roots)}")
print(f"  derived roots: {sorted(derived_roots)}")


✓ PASS — roots of x^3 - 6x^2 + 11x - 6 = 0
  your roots:    [1, 2, 3]
  derived roots: [1, 2, 3]


In [7]:
# Example: simplify/rationalise a surd expression and check equivalence
expr = 1 / (sqrt(5) - sqrt(3))
my_answer = (sqrt(5) + sqrt(3)) / 2

verify(expr, my_answer, label="rationalise 1/(√5 - √3)")


✓ PASS — rationalise 1/(√5 - √3)
  your answer:   sqrt(3)/2 + sqrt(5)/2
  derived form:  (sqrt(3) + sqrt(5))/2
  method:        symbolic simplify


True

## Sequences & series

Check a closed-form sum against SymPy's evaluation of the series.

In [8]:
# Example: sum of first n squares
series_expr = Sum(k**2, (k, 1, n)).doit()
my_formula = n*(n + 1)*(2*n + 1) / 6

verify(series_expr, my_formula, label="Σ k² for k=1..n")


✓ PASS — Σ k² for k=1..n
  your answer:   n*(n + 1)*(2*n + 1)/6
  derived form:  n*(2*n**2 + 3*n + 1)/6
  method:        symbolic simplify


True

In [9]:
# Example: geometric series sum, arithmetic progression, etc. — same pattern, swap the formula
arithmetic_sum = Sum(3 + (k - 1)*2, (k, 1, n)).doit()   # a=3, d=2
my_arith_formula = n * (n + 2)  # n/2 * (2a + (n-1)d) simplified for a=3, d=2

verify(arithmetic_sum, my_arith_formula, label="arithmetic series a=3, d=2")


✓ PASS — arithmetic series a=3, d=2
  your answer:   n*(n + 2)
  derived form:  n*(n + 2)
  method:        symbolic simplify


True

## Complex numbers

Verify modulus, argument, or algebraic manipulation of complex expressions.

In [10]:
# Example: (2 + 3i)(1 - i), then its modulus
z1 = (2 + 3*I) * (1 - I)
my_expanded = 5 + I
my_modulus = sqrt(26)

verify(sp.expand(z1), my_expanded, label="(2+3i)(1-i) expanded")
verify(Abs(z1), my_modulus, label="|(2+3i)(1-i)|")


✓ PASS — (2+3i)(1-i) expanded
  your answer:   5 + I
  derived form:  5 + I
  method:        symbolic simplify
✓ PASS — |(2+3i)(1-i)|
  your answer:   sqrt(26)
  derived form:  sqrt(26)
  method:        symbolic simplify


True

## Vectors

Check dot products, cross products, or magnitudes computed by hand.

In [11]:
from sympy import Matrix

u = Matrix([1, -2, 3])
v = Matrix([4, 0, -1])

my_dot = 1
my_cross = Matrix([2, 13, 8])
my_magnitude_u = sqrt(14)

print("dot product:")
verify(u.dot(v), my_dot, label="u · v")
print()
print("cross product (compared component-wise):")
derived_cross = u.cross(v)
match = list(derived_cross) == list(my_cross)
print(("\u2713 PASS" if match else "\u2717 FAIL") + " — u × v")
print(f"  your answer:   {list(my_cross)}")
print(f"  derived form:  {list(derived_cross)}")
print()
print("magnitude:")
verify(u.norm(), my_magnitude_u, label="|u|")


dot product:
✓ PASS — u · v
  your answer:   1
  derived form:  1
  method:        symbolic simplify

cross product (compared component-wise):
✓ PASS — u × v
  your answer:   [2, 13, 8]
  derived form:  [2, 13, 8]

magnitude:
✓ PASS — |u|
  your answer:   sqrt(14)
  derived form:  sqrt(14)
  method:        symbolic simplify


True

## Your own problem

Copy this cell for a new question — replace `expression` and `claimed_answer`, run it, and read the verdict.

In [12]:
expression = x**2 + 2*x + 1      # <- the thing you're deriving from (a derivative, an integral, an equation's LHS-RHS, ...)
claimed_answer = (x + 1)**2      # <- the answer you worked out by hand

verify(expression, claimed_answer, label="describe the problem here")


✓ PASS — describe the problem here
  your answer:   (x + 1)**2
  derived form:  x**2 + 2*x + 1
  method:        symbolic simplify


True